# Heimdall narrative analysis

Ingest is **persisted** in `heimdall.db` (or your `DATABASE_URL`). Each API ingest upserts posts and outrage scores.

```bash
pip install -e ".[notebook]"
```

Set `NARRATIVE_NAME` below (e.g. `midterms_2026`).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from heimdall.analysis.duplicates import find_duplicate_text_clusters
from heimdall.analysis.loader import load_narrative_posts, load_narratives

# --- configure ---
NARRATIVE_NAME = "midterms_2026"  # or set NARRATIVE_ID = 3
NARRATIVE_ID = None
MIN_OUTRAGE = 0.0

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "heimdall.db").exists() and (PROJECT_ROOT.parent / "heimdall.db").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

import os

os.chdir(PROJECT_ROOT)
DATABASE_URL = f"sqlite:///{(PROJECT_ROOT / 'heimdall.db').resolve()}"

print("Database:", DATABASE_URL)

In [ ]:
narratives = load_narratives(database_url=DATABASE_URL)
narratives

In [ ]:
posts = load_narrative_posts(
    narrative_id=NARRATIVE_ID,
    narrative_name=NARRATIVE_NAME if NARRATIVE_ID is None else None,
    database_url=DATABASE_URL,
    min_outrage=MIN_OUTRAGE if MIN_OUTRAGE > 0 else None,
)
print(f"Posts loaded: {len(posts)}")
posts.head(10)

In [ ]:
summary = {
    "narrative": posts["narrative_name"].iloc[0] if len(posts) else None,
    "posts": len(posts),
    "unique_authors": posts["author_id"].nunique(),
    "platforms": sorted(posts["platform"].dropna().unique().tolist()),
    "outrage_mean": round(posts["outrage_index"].mean(), 4) if posts["outrage_index"].notna().any() else None,
    "outrage_max": round(posts["outrage_index"].max(), 4) if posts["outrage_index"].notna().any() else None,
    "known_bots": int(posts["known_bot_label"].notna().sum()),
}
summary

In [ ]:
scored = posts.dropna(subset=["outrage_index"])
if scored.empty:
    print("No outrage scores yet — re-ingest or check narrative id.")
else:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(scored["outrage_index"], bins=20, color="#c0392b", edgecolor="white")
    ax.set_title("Outrage index distribution")
    ax.set_xlabel("outrage_index")
    ax.set_ylabel("posts")
    plt.tight_layout()
    plt.show()

    top = scored.nlargest(15, "outrage_index")[
        ["post_id", "author_handle", "outrage_index", "sentiment_label", "posted_at", "text"]
    ]
    from IPython.display import display

    display(top)

In [ ]:
clusters = find_duplicate_text_clusters(posts, min_posts=2)
print(f"Duplicate text clusters: {len(clusters)}")
for i, cluster in enumerate(clusters[:10], 1):
    print(f"\n--- cluster {i} ({cluster.count} posts, {len(cluster.author_ids)} authors) ---")
    print(cluster.sample_text)
    print("authors:", ", ".join(cluster.author_ids[:8]))

In [ ]:
repeat_authors = (
    posts.groupby(["author_id", "author_handle"], dropna=False)
    .size()
    .reset_index(name="post_count")
    .sort_values("post_count", ascending=False)
)
repeat_authors.head(15)

In [ ]:
if not posts.empty and posts["posted_at"].notna().any():
    daily = (
        posts.set_index("posted_at")
        .resample("D")
        .size()
        .rename("posts")
        .reset_index()
    )
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(daily["posted_at"], daily["posts"], marker="o")
    ax.set_title("Posts per day (ingested sample)")
    ax.set_ylabel("posts")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()
    daily